In [6]:
"""
Voting: Hard OR Soft Voting

Hard Voting: Each model votes for a class label and the class with the most votes is selected as the final prediction.

Soft Voting: Each model predicts the probability of each class label and the class with the highest average probability is selected as the final prediction.

*Weighted Voting: Each model is assigned a weight based on its performance, and the class with the highest weighted average probability is selected as the final prediction.
"""

'\nVoting: Hard OR Soft Voting\n\nHard Voting: Each model votes for a class label and the class with the most votes is selected as the final prediction.\n\nSoft Voting: Each model predicts the probability of each class label and the class with the highest average probability is selected as the final prediction.\n\n*Weighted Voting: Each model is assigned a weight based on its performance, and the class with the highest weighted average probability is selected as the final prediction.\n'

In [ ]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns 

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split,cross_val_score,StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import SVC 
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score , classification_report

In [3]:
data = load_breast_cancer()
features, target = data.data, data.target

In [4]:
Xtrain, Xtest, ytrain, ytest = train_test_split(features, target, test_size=0.2, random_state=42, stratify=target)

In [5]:
# Scale for SVM, Logistic, and KNN

scaler = StandardScaler()
Xtrain_scaled = scaler.fit_transform(Xtrain)
Xtest_scaled = scaler.fit_transform(Xtest)

In [15]:
# Defining Diverse base models

estimators = [
    ("lr", LogisticRegression(max_iter=1000,random_state=42)),
    ("rf", RandomForestClassifier(n_estimators=100,random_state=42)),
    ("svc", SVC(probability=True,random_state=42)),
    ("knn", KNeighborsClassifier(n_neighbors=5))
]
  


for name, model in estimators:
  # Only RF doesn't need scaled data
  X_use = Xtrain_scaled if name in ["lr", "svc", "knn"] else Xtrain
  scores = cross_val_score(model, X_use, ytrain, cv=5, scoring="accuracy")
  print(f"{name} - {scores.mean()}")

lr - 0.9802197802197803
rf - 0.953846153846154
svc - 0.9714285714285715
knn - 0.9670329670329669


c:\Users\moham\Downloads\My Learning\Ai&ML- Route\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\moham\Downloads\My Learning\Ai&ML- Route\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\moham\Downloads\My Learning\Ai&ML- Route\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\moham\Downloads\My Learning\Ai&ML- Route\.venv\Lib\site-packages\sklearn\

In [16]:
# Voting classifier 
# HARD VOTING

hard_voter = VotingClassifier(estimators=estimators, voting="hard")
hard_voter.fit(Xtrain_scaled, ytrain)

ypred_hard = hard_voter.predict(Xtest_scaled)

print(classification_report(ytest, ypred_hard))
print(accuracy_score(ytest, ypred_hard))

              precision    recall  f1-score   support

           0       0.98      0.98      0.98        42
           1       0.99      0.99      0.99        72

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114

0.9824561403508771


c:\Users\moham\Downloads\My Learning\Ai&ML- Route\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


In [ ]:
# Soft VOTING, we avg prob NOT Labels


soft_voter = VotingClassifier(estimators=estimators, voting="soft")
soft_voter.fit(Xtrain_scaled, ytrain)

ypred_soft = soft_voter.predict(Xtest_scaled)

print(classification_report(ytest, ypred_soft))
print(accuracy_score(ytest, ypred_soft))
# This is how we can get the probabilities of each class for each sample in the test set. The predict_proba method returns an array of shape (n_samples, n_classes) where each row corresponds to a sample and each column corresponds to a class. The values in the array represent the predicted probabilities of each class for each sample.probas = soft_voter.predict_proba(Xtest_scaled)
probas = soft_voter.predict_proba(Xtest_scaled)
for i,p in enumerate(probas):
    pred = data.target_names[np.argmax(p)]
    print(f"sample {i} malignant = {p[0]:.3f} benign {p[1]:.3f} -> pred {pred}")


              precision    recall  f1-score   support

           0       1.00      0.95      0.98        42
           1       0.97      1.00      0.99        72

    accuracy                           0.98       114
   macro avg       0.99      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114

0.9824561403508771
sample 0 malignant = 0.997 benign 0.003 -> pred malignant
sample 1 malignant = 0.000 benign 1.000 -> pred benign
sample 2 malignant = 0.953 benign 0.047 -> pred malignant
sample 3 malignant = 0.307 benign 0.693 -> pred benign
sample 4 malignant = 0.994 benign 0.006 -> pred malignant
sample 5 malignant = 0.027 benign 0.973 -> pred benign
sample 6 malignant = 0.000 benign 1.000 -> pred benign
sample 7 malignant = 0.997 benign 0.003 -> pred malignant
sample 8 malignant = 1.000 benign 0.000 -> pred malignant
sample 9 malignant = 0.986 benign 0.014 -> pred malignant
sample 10 malignant = 0.010 benign 0.990 -> pred benign
sample 11 malignant = 0.954 be

c:\Users\moham\Downloads\My Learning\Ai&ML- Route\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


In [ ]:
# Weighted Voting, also soft voting
"""
NOTE: all voters are equal , give more weights to stronger models
Important: Set weight from validation accuracy
"""

weighted_voter = VotingClassifier(estimators=estimators, voting="soft", weights=[1, 2, 2, 1])
# lr: 1, rf:2 , svc:2 , knn:1 
weighted_voter.fit(Xtrain_scaled, ytrain)

ypred_weighted = weighted_voter.predict(Xtest_scaled)

print(classification_report(ytest, ypred_weighted))
print(accuracy_score(ytest, ypred_weighted))

              precision    recall  f1-score   support

           0       1.00      0.95      0.98        42
           1       0.97      1.00      0.99        72

    accuracy                           0.98       114
   macro avg       0.99      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114

0.9824561403508771


c:\Users\moham\Downloads\My Learning\Ai&ML- Route\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
